# Caso 1: Predicción de retrasos en la entrega (Olist)

**Objetivo de la práctica:** armar, a partir de 6 tablas, un dataset donde una fila = un pedido, y crear la variable objetivo `is_delayed`.

## 1. Imports y ruta de los datos

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")  # carpeta con los CSV de Olist

## 2. Carga de las 6 tablas necesarias
Las columnas de fechas se leen directamente como fechas (`parse_dates`) para poder restar una fecha de otra.

In [2]:
# pedidos: de acá salen la variable objetivo, el mes y el día de la semana
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
    ],
)
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")        # destino
order_items = pd.read_csv(DATA / "olist_order_items_dataset.csv")    # tabla puente + flete
products = pd.read_csv(DATA / "olist_products_dataset.csv")          # peso y dimensiones
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv")            # origen
geolocation = pd.read_csv(DATA / "olist_geolocation_dataset.csv")    # coordenadas

# revisamos el tamaño (filas, columnas) de cada tabla
for nombre, tabla in [("orders", orders), ("customers", customers), ("order_items", order_items),
                      ("products", products), ("sellers", sellers), ("geolocation", geolocation)]:
    print(f"{nombre:12s} {tabla.shape}")

orders       (99441, 8)
customers    (99441, 5)
order_items  (112650, 7)
products     (32951, 9)
sellers      (3095, 4)
geolocation  (1000163, 5)


## 3. Filtrado de pedidos
Nos quedamos solo con los pedidos entregados y con fecha real de entrega conocida: sin ella no se puede calcular el retraso.

In [3]:
print("Pedidos antes del filtro:", len(orders))

# estado "delivered" y fecha real de entrega no nula
orders = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notna())
].copy()  # .copy() evita advertencias al modificar el resultado del filtro

print("Pedidos después del filtro:", len(orders))

Pedidos antes del filtro: 99441
Pedidos después del filtro: 96470


## 4. Variable objetivo y variables de fecha
- `dias_retraso`: cuántos días después de lo prometido llegó el pedido (un número negativo significa que llegó antes);
- `is_delayed`: 1 si `dias_retraso > 0`;
- `mes` y `dia_semana` (0 = lunes) salen de la fecha de compra.

In [4]:
# diferencia entre la fecha real y la estimada, expresada en días
orders["dias_retraso"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

# variable objetivo: True/False convertido a 1/0
orders["is_delayed"] = (orders["dias_retraso"] > 0).astype(int)

# variables temporales de la fecha de compra
orders["mes"] = orders["order_purchase_timestamp"].dt.month
orders["dia_semana"] = orders["order_purchase_timestamp"].dt.dayofweek

orders[["order_id", "dias_retraso", "is_delayed", "mes", "dia_semana"]].head()

,order_id,dias_retraso,is_delayed,mes,dia_semana
0,e481f51cbdc54678b7cc49136f2d6af7,-8,0,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,-6,0,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,-18,0,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,-13,0,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,-10,0,2,1


## 5. Desbalance de clases
Vemos qué proporción representan los retrasos. Esto define qué métricas usar (ROC-AUC, PR-AUC, F1).

In [5]:
# cantidad de pedidos en cada clase
print(orders["is_delayed"].value_counts())
print()
# proporción de cada clase
print(orders["is_delayed"].value_counts(normalize=True).round(4))

is_delayed
0    89936
1     6534
Name: count, dtype: int64

is_delayed
0    0.9323
1    0.0677
Name: proportion, dtype: float64


## 6. Coordenadas: una fila por código postal
En `geolocation` hay muchas filas por código postal. Calculamos el promedio de latitud y longitud para no multiplicar los pedidos al unir.

In [6]:
# una fila por código postal, con el promedio de lat y lng
geo_agg = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
    .reset_index()
)

# dos copias con nombres de columnas distintos: una para el cliente y otra para el vendedor
geo_cli = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "lat": "lat_cli", "lng": "lng_cli",
})
geo_ven = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "lat": "lat_ven", "lng": "lng_ven",
})

print("Códigos postales en geolocation:", geolocation["geolocation_zip_code_prefix"].nunique())
print("Filas en geo_agg:", len(geo_agg))  # debe coincidir con la línea anterior

Códigos postales en geolocation: 19015
Filas en geo_agg: 19015


## 7. Punto de destino: pedidos + clientes + coordenadas del cliente

In [7]:
destino = (
    # inner: cada pedido debe tener su cliente
    orders.merge(
        customers[["customer_id", "customer_zip_code_prefix", "customer_state"]],
        on="customer_id",
        how="inner",
    )
    # left: si falta la coordenada, conservamos el pedido (queda NaN)
    .merge(geo_cli, on="customer_zip_code_prefix", how="left")
)

print("Filas en destino:", len(destino))
destino[["order_id", "customer_state", "lat_cli", "lng_cli"]].head()

Filas en destino: 96470


,order_id,customer_state,lat_cli,lng_cli
0,e481f51cbdc54678b7cc49136f2d6af7,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,-23.676370,-46.514627


## 8. Ítems del pedido: productos, vendedores y coordenadas del vendedor
El volumen del producto es largo × alto × ancho.

In [8]:
# volumen en cm3 a partir de las tres dimensiones
products["volumen_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

items = (
    order_items[["order_id", "order_item_id", "product_id", "seller_id", "freight_value"]]
    # peso y volumen de cada producto
    .merge(products[["product_id", "product_weight_g", "volumen_cm3"]],
           on="product_id", how="left")
    # estado y código postal del vendedor
    .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]],
           on="seller_id", how="left")
    # coordenadas del vendedor
    .merge(geo_ven, on="seller_zip_code_prefix", how="left")
)

print("Ítems de pedidos:", len(items))
items.head()

Ítems de pedidos: 112650


,order_id,order_item_id,product_id,seller_id,freight_value,product_weight_g,volumen_cm3,seller_zip_code_prefix,seller_state,lat_ven,lng_ven
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,13.29,650.0,3528.0,27277,SP,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,19.93,30000.0,60000.0,3471,SP,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,17.87,3050.0,14157.0,37564,MG,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.79,200.0,2400.0,14403,SP,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,18.14,3750.0,42000.0,87900,PR,-22.929384,-53.135873


## 9. Distancia cliente-vendedor (fórmula de Haversine)
La función trabaja con columnas completas (vectorización de numpy), por eso no hace falta un bucle.

In [9]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Distancia en km entre dos puntos (latitud y longitud en grados)."""
    radio_tierra = 6371.0
    # pasamos de grados a radianes
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * radio_tierra * np.arcsin(np.sqrt(a))


# prueba de la función: São Paulo -> Río de Janeiro, esperamos unos 360 km
print(round(haversine_km(-23.55, -46.63, -22.91, -43.17)), "km")

361 km


In [10]:
# agregamos las coordenadas del cliente a cada ítem (por order_id)
items = items.merge(destino[["order_id", "lat_cli", "lng_cli"]], on="order_id", how="inner")

# distancia cliente-vendedor de cada ítem
items["distancia_km"] = haversine_km(
    items["lat_cli"], items["lng_cli"], items["lat_ven"], items["lng_ven"]
)

items["distancia_km"].describe()

count    109653.000000
mean        596.257743
std         588.552134
min           0.000000
25%         185.229335
50%         431.909443
75%         791.637120
max        8677.911622
Name: distancia_km, dtype: float64

## 10. Agregación: una fila por pedido
Sumamos flete, peso y volumen, y promediamos la distancia. Además contamos ítems y vendedores.

In [11]:
items_por_orden = items.groupby("order_id").agg(
    flete=("freight_value", "sum"),                # flete total del pedido
    peso_g=("product_weight_g", "sum"),            # peso total
    volumen_cm3=("volumen_cm3", "sum"),            # volumen total
    distancia_km=("distancia_km", "mean"),         # distancia promedio de los ítems
    n_items=("order_item_id", "count"),            # cantidad de ítems
    n_vendedores=("seller_id", "nunique"),         # cantidad de vendedores distintos
    seller_state=("seller_state", "first"),        # estado de origen (primer vendedor)
).reset_index()

items_por_orden.head()

,order_id,flete,peso_g,volumen_cm3,distancia_km,n_items,n_vendedores,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,13.29,650.0,3528.0,301.504681,1,1,SP
1,00018f77f2f0320c557190d7a144bdd3,19.93,30000.0,60000.0,585.563937,1,1,SP
2,000229ec398224ef6ca0657da4fc703e,17.87,3050.0,14157.0,312.343511,1,1,MG
3,00024acbcdf0a6daa1e931b038114c75,12.79,200.0,2400.0,293.168420,1,1,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,18.14,3750.0,42000.0,646.163463,1,1,PR


## 11. DataFrame final y verificaciones

In [12]:
# unimos el destino (una fila por pedido) con los datos agregados de los ítems
df = destino.merge(items_por_orden, on="order_id", how="inner")

print("Filas:", len(df), "| order_id únicos:", df["order_id"].nunique())
# si algún join duplicó pedidos, esta línea falla y avisa
assert len(df) == df["order_id"].nunique(), "Hay pedidos duplicados: revisar los joins"

# columnas con valores faltantes
print("\nValores faltantes por columna:")
print(df.isna().sum()[lambda s: s > 0])

Filas: 96470 | order_id únicos: 96470

Valores faltantes por columna:
order_approved_at                14
order_delivered_carrier_date      1
lat_cli                         264
lng_cli                         264
distancia_km                    476
dtype: int64


## 12. Columnas para el modelo (sin fuga de datos)
Quitamos la fecha real de entrega y `dias_retraso`: en el momento de la compra todavía no existen.

In [13]:
columnas_modelo = [
    "is_delayed",      # variable objetivo
    "distancia_km",
    "peso_g",
    "volumen_cm3",
    "customer_state",  # estado de destino
    "seller_state",    # estado de origen
    "mes",
    "dia_semana",
    "flete",
]
df_modelo = df[columnas_modelo].copy()

print(df_modelo.shape)
df_modelo.head()

(96470, 9)


,is_delayed,distancia_km,peso_g,volumen_cm3,customer_state,seller_state,mes,dia_semana,flete
0,0,18.576110,500.0,1976.0,SP,SP,10,0,8.72
1,0,851.495069,400.0,4693.0,BA,SP,7,1,22.76
2,0,514.410666,420.0,9576.0,GO,SP,8,2,19.22
3,0,1822.226336,450.0,6000.0,RN,MG,11,5,27.20
4,0,29.676625,250.0,11475.0,SP,SP,2,1,8.72


## 13. Conclusiones

- **Filtrado:** de los 99 441 pedidos originales se conservaron 96 470 (los pedidos con estado `delivered` y fecha real de entrega). Se descartaron 2 971 pedidos (3,0 %).
- **Dataset final:** 96 470 filas (una por pedido, sin duplicados) y 9 columnas: la variable objetivo `is_delayed` y 8 variables predictoras.
- **Desbalance de clases:** el 6,77 % de los pedidos llegó con retraso (6 534) y el 93,23 % a tiempo (89 936), una relación de aproximadamente 14 a 1. Un modelo que siempre prediga 0 alcanzaría 93,23 % de accuracy sin detectar ningún retraso (recall = 0, F1 = 0), por lo que accuracy no es una métrica útil. Evaluaremos con ROC-AUC, PR-AUC y F1-Score.
- **Valores faltantes:** solo `distancia_km` tiene faltantes en las variables del modelo (476 pedidos, 0,49 %), debido a códigos postales sin coordenadas en `geolocation`. `peso_g` y `volumen_cm3` no tienen faltantes. Decisión: eliminar esas 476 filas, porque son menos del 1 %.
- **Valores atípicos en la distancia:** la distancia máxima es de 8 678 km, mayor que cualquier distancia posible dentro de Brasil (aprox. 4 400 km). Esto indica coordenadas erróneas en `geolocation`. Pendiente: revisar y filtrar las coordenadas fuera de Brasil.
- **Fuga de datos:** se excluyeron `order_delivered_customer_date` y `dias_retraso`, porque no se conocen al momento de la compra.
- **Próximo paso:** dividir en train/test con `stratify=y`, codificar `customer_state` y `seller_state`, y entrenar el baseline, una regresión logística (con `class_weight="balanced"` por el desbalance).